# **Wine Quality Classification — Gaussian Naive Bayes**

Dalam pengerjaan UTS ini, saya tidak menggunakan library instan seperti scikit-learn agar logika di balik algoritma Gaussian Naive Bayes bisa terlihat transparan. Berikut adalah langkah-langkah yang saya lakukan:

1. Tahap Persiapan Data: Saya mulai dengan membaca file CSV menggunakan modul csv standar. Karena target kualitas wine adalah nilai numerik (skor), saya melakukan binning (pelabelan biner) di mana skor ≥6 saya tandai sebagai 1 (Bagus/Ekspor) dan sisanya sebagai 0 (Biasa/Lokal).

2. Perhitungan Statistik Dasar: Untuk memenuhi asumsi Gaussian, saya memisahkan dataset menjadi dua kelompok berdasarkan kelasnya. Kemudian, saya menghitung nilai rata-rata (μ) dan standar deviasi (σ) untuk setiap 11 fitur kimia pada masing-masing kelas.

3. Implementasi Model Matematika: Saya membangun fungsi gaussian_pdf untuk menghitung peluang distribusi normal. Dengan fungsi ini, saya dapat menentukan seberapa "cocok" nilai fitur baru terhadap karakteristik wine Bagus maupun Biasa.

4. Prediksi & Pengambilan Keputusan: Saya menggabungkan peluang prior (frekuensi kemunculan kelas) dengan hasil kali seluruh likelihood fitur. Akhirnya, saya membuat fungsi predict yang akan memilih kelas dengan nilai probabilitas posterior tertinggi.

5. Validasi & Analisis: Sebagai langkah akhir, saya menguji model dengan data baru. Saya juga melakukan analisis sensitivitas dengan mengubah variabel input (seperti kadar alkohol) untuk melihat seberapa besar pengaruhnya terhadap pergeseran probabilitas akhir.

Dengan proses yang saya lakukan ini, diharapkan nya dapat membuat sistem klasifikasi yang tidak hanya akurat, tetapi juga dapat dijelaskan logika probabilitasnya langkah demi langkah.

# **CELL 1 — Import Library**

In [223]:
import csv # Untuk membaca file CSV
import math # Untuk operasi matematika: sqrt, exp, pi

# **CELL 2 — Load & Eksplorasi Dataset**

Ngebaca file CSV dan mengembalikan header + data mentah.

Parameter:
filepath (str): Path ke file CSV

In [224]:
def load_dataset(filepath):

    dataset = [] # Tempat nyimpen semua baris data

    with open(filepath, 'r') as f: # Buka file CSV
        reader = csv.reader(f) # Buat reader objek
        headers = next(reader) # Baris pertama = nama kolom

        for row in reader: # Iterasi tiap baris data
            if row: # Abaikan baris kosong
                dataset.append(row) # Simpan baris ke list

    return headers, dataset

# Jalankan fungsi loadnya
headers, raw_data = load_dataset('winequalityN.csv')

# Cek datasetnya
print(f"Jumlah baris  : {len(raw_data)}")
print(f"Jumlah kolom  : {len(headers)}")
print(f"Nama kolom    : {headers}")

Jumlah baris  : 6497
Jumlah kolom  : 13
Nama kolom    : ['type', 'fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'quality']


# **CELL 3 — Pra-pemrosesan: Labeling Biner & Konversi Tipe Data**

Ngelakuin pra-pemrosesan data:
1. Abaikan kolom 'type' (non-numerik)
2. Konversi semua fitur ke float
3. Lalu ubah si kolom 'quality' jadi label biner:
    - quality >= 6  berarti  1 (Bagus / layak ekspor)
    - quality <  6  berarti  0 (Biasa / pasar lokal)

In [225]:

def preprocess(raw_data):

    processed = [] # Tempat menyimpan data yang sudah diproses

    skipped = 0 # Konter baris yang dilewatin karena missing value

    for row in raw_data:
    # 0=type, 1=fixed acidity, 2=volatile acidity, 3=citric acid,
    # 4=residual sugar, 5=chlorides, 6=free SO2, 7=total SO2,
    # 8=density, 9=pH, 10=sulphates, 11=alcohol, 12=quality

        # Ngecek apakah ada nilai yang kosong (missing value) di kolom fitur atau target
        if any(row[i].strip() == '' for i in range(1, 13)):
            skipped += 1 # Lewatin baris yang memiliki nilai kosong
            continue

        features = [float(row[i]) for i in range(1, 12)]

        # Ambil skor kualitas dari kolom terakhir (berarti indeks 12)
        quality_score = float(row[12])

        label = 1 if quality_score >= 6 else 0 # Konversi ke biner, kalau >= 6 bagus, kalau <6 biasa

        # Ngegabungin fitur dan label jadi 1 baris, lalu save
        processed.append(features + [label])

    print(f"Baris yang dilewati (missing value): {skipped}")

    return processed

# Jalankan pra pemrosesan
data = preprocess(raw_data)

# Itung distribusi kelasnya]
kelas_0 = sum(1 for row in data if row[-1] == 0)
kelas_1 = sum(1 for row in data if row[-1] == 1)

print("Pra-Pemrosesan selesai")
print(f"   Total data      : {len(data)} sampel")
print(f"   Kelas 0 (Biasa) : {kelas_0} sampel ({kelas_0/len(data)*100:.1f}%)")
print(f"   Kelas 1 (Bagus) : {kelas_1} sampel ({kelas_1/len(data)*100:.1f}%)")

Baris yang dilewati (missing value): 34
Pra-Pemrosesan selesai
   Total data      : 6463 sampel
   Kelas 0 (Biasa) : 2372 sampel (36.7%)
   Kelas 1 (Bagus) : 4091 sampel (63.3%)


# **CELL 4 — Fungsi: separate_by_class(dataset)**

Mengelompokkan baris data berdasarkan label kelasnya.

Parameter:
    dataset (list): Data yang sudah diproses (fitur + label)

In [226]:
def separate_by_class(dataset):

  separated = {} # Untuk nampung data per kelas

  for row in dataset:
    label = int(row[-1]) # Ambil elememn terakhir tiap baris

    if label not in separated:
      separated[label] = []  # JIKA kelas belum ada di dict, bikin list kosong untuk kelas tsb

    separated[label].append(row[:-1]) # Save tapi fitunya aja tanpa label, kedalam kelas yang sesuai

  return separated

# RUn pemisahan kelasny
separated = separate_by_class(data)

print(f"Pemisahan kelas selesai")
for kelas, rows in separated.items():
    print(f"   Kelas {kelas}: {len(rows)} baris")

Pemisahan kelas selesai
   Kelas 1: 4091 baris
   Kelas 0: 2372 baris


# **CELL 5 — Fungsi: summarize_by_class(). Menghitung Mean & Standar Deviasi per fitur per kelas**

In [227]:
def mean(values): # Itung nilai rata-rata dari sebuah list angka
  return sum(values) / len(values) # Jumlah semua nilai dibagi banyaknya data

def stdev(values): # Itung standar deviasi dari sebuah list angka
  avg = mean(values) # Itung rata-ratanya dulu
  variance = sum((x-avg) **2 for x in values) # Jumlah kuadrat selisih
  variance = variance / (len(values) - 1) # Bagi n-1
  return math.sqrt(variance)

def summarize(features_list): # Ngerangkum statistikny
  # Transpose matrix dari baris ke kolom, jadi bisa ngitung statistik per kolom
  summaries = [
      (mean(col), stdev(col), len(col))
      for col in zip(*features_list)
  ]
  return summaries

def summarize_by_class(separated): # Itung statistik utk tiap fitur pada masing masing klas
  summaries = {}

  for kelas, rows in separated.items():
    summaries[kelas] = summarize(rows) # Panggil summarize() untuk setiap kelas

  return summaries

# Run perhitungan statistik
summaries = summarize_by_class(separated)

feature_names = [
    'fixed acidity', 'volatile acidity', 'citric acid',
    'residual sugar', 'chlorides', 'free sulfur dioxide',
    'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol'
]

print("Statistik per kelas berhasil dihitung\n")
for kelas, stats in summaries.items():
    label_name = "Bagus (1)" if kelas == 1 else "Biasa (0)"
    print(f"Kelas {kelas} — {label_name}")
    print(f"   {'Fitur':<25} {'Mean (μ)':>12} {'Std Dev (σ)':>12}")
    print(f"   {'-'*50}")
    for i, (mu, sigma, _) in enumerate(stats):
        print(f"   {feature_names[i]:<25} {mu:>12.4f} {sigma:>12.4f}")
    print()

Statistik per kelas berhasil dihitung

Kelas 1 — Bagus (1)
   Fitur                         Mean (μ)  Std Dev (σ)
   --------------------------------------------------
   fixed acidity                   7.1515       1.3101
   volatile acidity                0.3060       0.1387
   citric acid                     0.3271       0.1332
   residual sugar                  5.3349       4.6612
   chlorides                       0.0512       0.0285
   free sulfur dioxide            31.1166      16.3932
   total sulfur dioxide          113.6549      53.0648
   density                         0.9941       0.0031
   pH                              3.2205       0.1605
   sulphates                       0.5352       0.1516
   alcohol                        10.8516       1.2192

Kelas 0 — Biasa (0)
   Fitur                         Mean (μ)  Std Dev (σ)
   --------------------------------------------------
   fixed acidity                   7.3320       1.2688
   volatile acidity                0.3975 

# **CELL 6 — Fungsi: calculate_prior().Probabilitas kemunculan masing-masing kelas**

Ngitung probabilitas prior P(Ck) untuk setiap kelas

`Rumus: P(Ck) = jumlah sampel kelas k / total sampel`

In [228]:
def calculate_prior(separated):

  # Itung total seluruh data (semua kelas dijumlahi)
  total = sum(len(rows) for rows in separated.values())

  prior = {}
  for kelas, rowsn in separated.items():
    prior[kelas] = len(rows)/ total # Proporsi data kelas ini ditotalin

  return prior

# Itung prior
prior = calculate_prior(separated)

print("Probabilitas prior:")
for kelas, prob in prior.items():
    label_name = "Bagus (1)" if kelas == 1 else "Biasa (0)"
    print(f"   Kelas {kelas} — {label_name}: {prob:.4f}")

Probabilitas prior:
   Kelas 1 — Bagus (1): 0.3670
   Kelas 0 — Biasa (0): 0.3670


# **CELL 7 — Fungsi: gaussian_pdf(). Likelihood — Distribusi Gaussian**

Ngitung nilai probabilitas satu fitur menggunakan rumus Probability Density Function (PDF) distribusi Gaussian (Normal).

Rumus:
`P(x | C) = (1 / sqrt(2π σ²)) × exp( -(x - μ)² / (2σ²) )`

In [229]:
def gaussian_pdf(x, mean, stdev):

    exponent = math.exp( # Itung bagian eksponensialny
        -((x - mean) ** 2) / # -(x - μ)²
        (2 * stdev ** 2) # dibagi dengan 2σ²
    )

    # Itung koefisien normalisasi: 1 / (√(2π) × σ)
    coefficient = 1 / (math.sqrt(2 * math.pi) * stdev)

    # Gabungin koefisien × eksponensial = nilai PDF
    return coefficient * exponent

# **CELL 8 — Fungsi: calculate_posterior(). Skor akhir tiap kelas = Prior × ∏ Likelihood**

Ngitung probabilitas posterior untuk setiap kelas berdasarkan fitur input yang diberikan.

Naive Bayes mengasumsikan fitur saling independen, sehingga:
`P(C | X) ∝ P(C) × P(x1|C) × P(x2|C) × ... × P(x11|C)`

In [230]:
def calculate_posterior(summaries, prior, input_features):

    posteriors = {}

    for kelas, stats in summaries.items():

        # Mulai dengan log(Prior)
        log_posterior = math.log(prior[kelas]) # Make logaritma natural untuk kestabilan numerik

        for i, (mu, sigma, _) in enumerate(stats):
            x = input_features[i] # Ambil nilai fitur ke-i dari data uji

            likelihood = gaussian_pdf(x, mu, sigma)# Itung P(xi | C) menggunakan Gaussian PDF yang tadi

            if likelihood > 0:
                log_posterior += math.log(likelihood) # Tambahkan log(likelihood) ke akumulasi log-posterior
            else:
                log_posterior += math.log(1e-300) # Pake nilai sangat kecil untk pengganti log(0)

        # Simpan skor log-posterior untuk kelas ini
        posteriors[kelas] = log_posterior

    return posteriors

# **CELL 9 — Fungsi: predict() & evaluate_accuracy()**

Ngeprediksi kelas untuk satu baris data input.

Cara kerjanya:
1. Itung posterior untuk semua kelas
2. Pilih kelas dengan posterior tertinggi (argmax)

In [231]:
def predict(summaries, prior, input_features):

    # Itung skor log-posterior untuk semua kelas
    posteriors = calculate_posterior(summaries, prior, input_features)

    return max(posteriors, key=posteriors.get)

def evaluate_accuracy(summaries, prior, dataset): # Mengevaluasi akurasi model dengan membandingkan prediksi terhadap label asli di seluruh dataset

    correct = 0 # Konter prediksi benar

    for row in dataset:
        features = row[:-1] # Ngambik 11 fitur (tanpa label)
        true_label = row[-1] # Ngambik label asli

        predicted = predict(summaries, prior, features) # Prediksi kelas untuk baris ini

        if predicted == true_label:
            correct += 1# Tambah konter kalo prediksi sesuai label asli

    accuracy = (correct / len(dataset)) * 100 # Itung persentase akurasi = benar / total × 100%

    return accuracy

accuracy = evaluate_accuracy(summaries, prior, data) # Evaluasi akurasi model di seluruh dataset
print(f"\nAkurasi Model Gaussian Naive Bayes: {accuracy:.2f}%")
print(f"   (Benar memprediksi {accuracy/100*len(data):.0f} dari {len(data)} sampel)")


Akurasi Model Gaussian Naive Bayes: 68.00%
   (Benar memprediksi 4395 dari 6463 sampel)


# **CELL 10 — Uji Data Baru (Pertanyaan a & b)**

In [232]:
# Data uji yang ad di soal
sample = {
    'fixed acidity'       : 8.2,
    'volatile acidity'    : 0.45,
    'citric acid'         : 0.17,
    'residual sugar'      : 19.2,
    'chlorides'           : 0.034,
    'free sulfur dioxide' : 19,
    'total sulfur dioxide': 172,
    'density'             : 0.993,
    'pH'                  : 3.12,
    'sulphates'           : 0.4,
    'alcohol'             : 10.1
}

# Konversi ke list (nah urutan harus sama dengan fitur saat training)
sample_features = list(sample.values())

# Prediksi kelasny
result = predict(summaries, prior, sample_features)
label_text = "BAGUS (Kelas 1) — Layak Ekspor" if result == 1 else "BIASA (Kelas 0) — Pasar Lokal"

print("=" * 60)
print("HASIL PREDIKSI DATA UJI BARU") # Header
print("=" * 60)
for feat, val in sample.items():
    print(f"   {feat:<25}: {val}")
print(f"\nPrediksi Kelas : {label_text}")

# Nampilin detail posterior tiap kelas
posteriors = calculate_posterior(summaries, prior, sample_features)
print(f"\nSkor Log-Posterior:")
for kelas, score in posteriors.items():
    label = "Bagus (1)" if kelas == 1 else "Biasa (0)"
    print(f"   Kelas {kelas} [{label}]: {score:.4f}")

# Analisis Likelihood per fitur (Jawaban Pertanyaan b)
print("\n" + "=" * 60)
print("ANALISIS LIKELIHOOD PER FITUR (Pertanyaan b)")
print("=" * 60)
print(f"   {'Fitur':<25} {'P(x|Kelas=0)':>15} {'P(x|Kelas=1)':>15} {'Favorit'}")
print(f"   {'-'*65}")

for i, (feat, val) in enumerate(sample.items()):
    mu0, sig0, _ = summaries[0][i] # Statistik kelas 0
    mu1, sig1, _ = summaries[1][i] # Statistik kelas 1

    lik0 = gaussian_pdf(val, mu0, sig0) # Likelihood kelas 0
    lik1 = gaussian_pdf(val, mu1, sig1) # Likelihood kelas 1

    favor = "-> Kelas 1 ✅" if lik1 > lik0 else "-> Kelas 0 ⚠️"
    print(f"   {feat:<25} {lik0:>15.6f} {lik1:>15.6f} {favor}")

HASIL PREDIKSI DATA UJI BARU
   fixed acidity            : 8.2
   volatile acidity         : 0.45
   citric acid              : 0.17
   residual sugar           : 19.2
   chlorides                : 0.034
   free sulfur dioxide      : 19
   total sulfur dioxide     : 172
   density                  : 0.993
   pH                       : 3.12
   sulphates                : 0.4
   alcohol                  : 10.1

Prediksi Kelas : BIASA (Kelas 0) — Pasar Lokal

Skor Log-Posterior:
   Kelas 1 [Bagus (1)]: -10.7325
   Kelas 0 [Biasa (0)]: -9.8642

ANALISIS LIKELIHOOD PER FITUR (Pertanyaan b)
   Fitur                        P(x|Kelas=0)    P(x|Kelas=1) Favorit
   -----------------------------------------------------------------
   fixed acidity                    0.248822        0.221069 -> Kelas 0 ⚠️
   volatile acidity                 2.039780        1.677726 -> Kelas 0 ⚠️
   citric acid                      1.742538        1.494030 -> Kelas 0 ⚠️
   residual sugar                   0.001792  

# **CELL 11 — Analisis Sensitivitas: Ubah Nilai Alcohol (Pertanyaan c)**

In [233]:
print("\n" + "=" * 60)
print("ANALISIS SENSITIVITAS — Ngubah Kadar Alcohol (Pertanyaan c)")
print("=" * 60)

# Daftar nilai alcohol yang akan dicoba
alcohol_values = [8.0, 9.0, 10.1, 11.0, 12.0, 13.0, 14.0, 14.9]

print(f"\n   {'Alcohol':>10} {'Prediksi':<20} {'Skor Kelas 0':>15} {'Skor Kelas 1':>15}")
print(f"   {'-'*65}")

for alc in alcohol_values:
    # Bikin dulu salinan fitur dan ganti nilai alcohol (indeks ke-10)
    modified_features = sample_features.copy()
    modified_features[10] = alc # Indeks 10 = alcohol

    pred = predict(summaries, prior, modified_features)
    posts = calculate_posterior(summaries, prior, modified_features)

    pred_label = "BAGUS ✅" if pred == 1 else "BIASA ⚠️"
    marker = " < (original)" if alc == 10.1 else ""

    print(f"   {alc:>10.1f} {pred_label:<20} {posts[0]:>15.4f} {posts[1]:>15.4f}{marker}")

print(f"\nKESIMPULAN:")
print(f"   Semakin tinggi kadar alcoholnya, semakin besar pula peluang wine")
print(f"   diklasifikasikan sebagai BAGUS (Kelas 1), karena wine dengan")
print(f"   kualitas tinggi cenderung memiliki kadar alkohol lebih tinggi.")


ANALISIS SENSITIVITAS — Ngubah Kadar Alcohol (Pertanyaan c)

      Alcohol Prediksi                Skor Kelas 0    Skor Kelas 1
   -----------------------------------------------------------------
          8.0 BIASA ⚠️                    -12.2998        -13.2778
          9.0 BIASA ⚠️                    -10.3659        -11.6957
         10.1 BIASA ⚠️                     -9.8642        -10.7325 < (original)
         11.0 BAGUS ✅                     -10.7205        -10.5499
         12.0 BAGUS ✅                     -13.0091        -10.9861
         13.0 BAGUS ✅                     -16.7052        -12.0950
         14.0 BAGUS ✅                     -21.8088        -13.8767
         14.9 BAGUS ✅                     -27.6055        -16.0555

KESIMPULAN:
   Semakin tinggi kadar alcoholnya, semakin besar pula peluang wine
   diklasifikasikan sebagai BAGUS (Kelas 1), karena wine dengan
   kualitas tinggi cenderung memiliki kadar alkohol lebih tinggi.
